In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Set Up Folders

input_folder  = "Final_Enhanced/"
gt_folder     = "GroundTruth_Potholes/"
output_folder = "Member1_Segmentation/"

os.makedirs(output_folder, exist_ok=True)

# Read Pothole Frame Name

pothole_input = "Frames_Potholes/"

all_files = os.listdir(pothole_input)

frame_files = []
for f in all_files:
    if f.endswith(".jpg"):
        frame_files.append(f)

frame_files.sort(key=lambda x: int(x.split("_")[1].split(".")[0]))

print("Total pothole frames found:", len(frame_files))

# Check Ground Truth Masks

frames_with_gt = []
for f in frame_files:
    if os.path.exists(gt_folder + f):
        frames_with_gt.append(f)

print("Frames with ground truth masks:", len(frames_with_gt))

if len(frames_with_gt) == 0:
    print("WARNING: No ground truth masks found in", gt_folder)


# Settings

ROI_80_89   = (420, 180, 780, 440)
ROI_195_199 = (380, 180, 720, 420)

THRESHOLD_WET = 160
THRESHOLD_DRY = 80

MIN_AREA = 800


# Per-frame kernel size settings

def get_frame_settings(frame_name):
    num = int(frame_name.split("_")[1].split(".")[0])

    if num >= 80 and num <= 89:
        # Wet pothole: detect bright water reflection
        return ROI_80_89, THRESHOLD_WET, "BINARY", 11

    elif num == 195:
        # Dry pothole, single dark region
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 21

    elif num == 196 or num == 197:
        # Two dark patches close together - need bigger kernel to merge
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 31

    else:
        # Frame 198, 199 - two dark patches with brighter gap
        # Use largest kernel + merge strategy
        return ROI_195_199, THRESHOLD_DRY, "BINARY_INV", 41


# Segmentation Function

def segment_pothole_frame(img, frame_name):

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    roi, threshold, mode, kernel_size = get_frame_settings(frame_name)
    x1, y1, x2, y2 = roi

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(roi_mask, (x1, y1), (x2, y2), 255, -1)

    gray_roi = cv2.bitwise_and(gray, gray, mask=roi_mask)

    if mode == "BINARY":
        _, binary = cv2.threshold(gray_roi, threshold, 255, cv2.THRESH_BINARY)
    else:
        _, binary = cv2.threshold(gray_roi, threshold, 255, cv2.THRESH_BINARY_INV)
        binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)

    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    final_mask = cv2.dilate(cleaned, kernel_dilate, iterations=2)

    return final_mask, gray, binary, cleaned, roi


def get_best_contours(final_mask, frame_name):
    num = int(frame_name.split("_")[1].split(".")[0])

    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    good = [c for c in contours if cv2.contourArea(c) > MIN_AREA]

    if len(good) == 0:
        return []

    # For frames 198 and 199: if multiple contours detected,
    # merge them all into one single convex hull
    # This handles the case where the pothole has two dark patches
    # with a bright sandy gap between them
    if num >= 198 and len(good) > 1:
        all_points = np.vstack(good)
        merged_hull = cv2.convexHull(all_points)
        return [merged_hull]

    # For all other frames: apply convex hull to each contour
    result = []
    for c in good:
        hull = cv2.convexHull(c)
        result.append(hull)

    return result


# Test on First Frame

print("\nTesting on first frame...")

test_name = frame_files[0]
test_img  = cv2.imread(pothole_input + test_name)

if test_img is None:
    print("Could not read the first frame. Check the folder path.")
else:

    final_mask, gray, binary, cleaned, roi = segment_pothole_frame(test_img, test_name)
    contours = get_best_contours(final_mask, test_name)

    marked_test = test_img.copy()
    cv2.drawContours(marked_test, contours, -1, (0, 0, 255), 2)

    x1, y1, x2, y2 = roi
    cv2.rectangle(marked_test, (x1, y1), (x2, y2), (0, 255, 255), 1)

    clean_mask = np.zeros(test_img.shape[:2], dtype=np.uint8)
    cv2.drawContours(clean_mask, contours, -1, 255, -1)

    plt.figure(figsize=(18, 3))

    plt.subplot(1, 6, 1)
    plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    plt.title("Original Frame")
    plt.axis('off')

    plt.subplot(1, 6, 2)
    plt.imshow(gray, cmap='gray')
    plt.title("Grayscale")
    plt.axis('off')

    plt.subplot(1, 6, 3)
    plt.imshow(binary, cmap='gray')
    plt.title("Threshold\ninside ROI")
    plt.axis('off')

    plt.subplot(1, 6, 4)
    plt.imshow(cleaned, cmap='gray')
    plt.title("After Closing\n+ Opening")
    plt.axis('off')

    plt.subplot(1, 6, 5)
    plt.imshow(clean_mask, cmap='gray')
    plt.title("Final Mask")
    plt.axis('off')

    plt.subplot(1, 6, 6)
    plt.imshow(cv2.cvtColor(marked_test, cv2.COLOR_BGR2RGB))
    plt.title("Marked Output\n(yellow = ROI)")
    plt.axis('off')

    plt.suptitle("Pothole Segmentation Final (" + test_name + ")")
    plt.tight_layout()
    plt.savefig(output_folder + "test_steps_" + test_name)
    plt.show()

    print("Test done. Check:", output_folder + "test_steps_" + test_name)
    print("Contours found:", len(contours))


# Process All Frames

print("\nProcessing all pothole frames...")

for frame_name in frame_files:

    img = cv2.imread(pothole_input + frame_name)

    if img is None:
        print("Could not read:", frame_name, "- Skipping.")
        continue

    final_mask, gray, binary, cleaned, roi = segment_pothole_frame(img, frame_name)
    contours = get_best_contours(final_mask, frame_name)

    marked = img.copy()
    cv2.drawContours(marked, contours, -1, (0, 0, 255), 2)

    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.drawContours(clean_mask, contours, -1, 255, -1)

    cv2.imwrite(output_folder + "mask_"       + frame_name, clean_mask)
    cv2.imwrite(output_folder + "marked_"     + frame_name, marked)

    enhanced_bgr = cv2.imread(input_folder + frame_name)
    if enhanced_bgr is None:
        enhanced_bgr = img

    mask_bgr   = cv2.cvtColor(clean_mask, cv2.COLOR_GRAY2BGR)
    comparison = np.hstack((enhanced_bgr, mask_bgr, marked))
    cv2.imwrite(output_folder + "comparison_" + frame_name, comparison)

print("Done. All outputs saved in:", output_folder)


# Calculate Evaluation Metrics

print("\n--- Evaluation Metrics (Frames with Ground Truth) ---")
print("Frame                  Accuracy  Precision  Recall    F1        IoU       Dice")
print("-" * 90)

results = []

for frame_name in frames_with_gt:

    img    = cv2.imread(pothole_input + frame_name)
    gt_img = cv2.imread(gt_folder + frame_name, cv2.IMREAD_GRAYSCALE)

    if img is None or gt_img is None:
        print("Could not read:", frame_name, "- Skipping.")
        continue

    final_mask, _, _, _, _ = segment_pothole_frame(img, frame_name)
    contours = get_best_contours(final_mask, frame_name)

    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.drawContours(clean_mask, contours, -1, 255, -1)

    gt_img = cv2.resize(gt_img, (clean_mask.shape[1], clean_mask.shape[0]))

    _, gt_bin   = cv2.threshold(gt_img,     127, 1, cv2.THRESH_BINARY)
    _, pred_bin = cv2.threshold(clean_mask, 127, 1, cv2.THRESH_BINARY)

    TP = int(np.sum((pred_bin == 1) & (gt_bin == 1)))
    FP = int(np.sum((pred_bin == 1) & (gt_bin == 0)))
    FN = int(np.sum((pred_bin == 0) & (gt_bin == 1)))
    TN = int(np.sum((pred_bin == 0) & (gt_bin == 0)))

    total = TP + FP + FN + TN

    accuracy = (TP + TN) / total

    if (TP + FP) == 0:
        precision = 0.0
    else:
        precision = TP / (TP + FP)

    if (TP + FN) == 0:
        recall = 0.0
    else:
        recall = TP / (TP + FN)

    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    if (TP + FP + FN) == 0:
        iou = 0.0
    else:
        iou = TP / (TP + FP + FN)

    if (2 * TP + FP + FN) == 0:
        dice = 0.0
    else:
        dice = (2 * TP) / (2 * TP + FP + FN)

    results.append([frame_name, accuracy, precision, recall, f1, iou, dice])

    print(frame_name, "\t",
          round(accuracy,  4), "\t",
          round(precision, 4), "\t",
          round(recall,    4), "\t",
          round(f1,        4), "\t",
          round(iou,       4), "\t",
          round(dice,      4))


# Averages and Save Metrics

if len(results) > 0:

    total_acc = 0
    total_pre = 0
    total_rec = 0
    total_f1  = 0
    total_iou = 0
    total_dic = 0

    for r in results:
        total_acc = total_acc + r[1]
        total_pre = total_pre + r[2]
        total_rec = total_rec + r[3]
        total_f1  = total_f1  + r[4]
        total_iou = total_iou + r[5]
        total_dic = total_dic + r[6]

    count = len(results)

    avg_acc = total_acc / count
    avg_pre = total_pre / count
    avg_rec = total_rec / count
    avg_f1  = total_f1  / count
    avg_iou = total_iou / count
    avg_dic = total_dic / count

    print("-" * 90)
    print("AVERAGE", "\t\t\t",
          round(avg_acc, 4), "\t",
          round(avg_pre, 4), "\t",
          round(avg_rec, 4), "\t",
          round(avg_f1,  4), "\t",
          round(avg_iou, 4), "\t",
          round(avg_dic, 4))

    result_file = open(output_folder + "pothole_metrics.txt", "w")
    result_file.write("Pothole Segmentation - Evaluation Results (Final Version)\n")
    result_file.write("Member 1 - ICT2403 Graphics and Image Processing\n")
    result_file.write("Degradation Type: Pothole\n")
    result_file.write("Approach: ROI + Bright/Dark detection + Per-frame kernel + Merge strategy\n")
    result_file.write("ROI Frames 80-89:    " + str(ROI_80_89) + "\n")
    result_file.write("ROI Frames 195-199:  " + str(ROI_195_199) + "\n")
    result_file.write("Threshold Wet (80-89):   " + str(THRESHOLD_WET) + "\n")
    result_file.write("Threshold Dry (195-199): " + str(THRESHOLD_DRY) + "\n")
    result_file.write("Kernel sizes: 80-89=11, 195=21, 196-197=31, 198-199=41\n")
    result_file.write("Total frames tested: " + str(len(frame_files)) + "\n")
    result_file.write("Frames with ground truth: " + str(len(results)) + "\n")
    result_file.write("-" * 90 + "\n")
    result_file.write("Frame                  Accuracy  Precision  Recall    F1        IoU       Dice\n")
    result_file.write("-" * 90 + "\n")

    for r in results:
        result_file.write(r[0] + "\t" +
                          str(round(r[1], 4)) + "\t" +
                          str(round(r[2], 4)) + "\t" +
                          str(round(r[3], 4)) + "\t" +
                          str(round(r[4], 4)) + "\t" +
                          str(round(r[5], 4)) + "\t" +
                          str(round(r[6], 4)) + "\n")

    result_file.write("-" * 90 + "\n")
    result_file.write("AVERAGE\t\t\t" +
                      str(round(avg_acc, 4)) + "\t" +
                      str(round(avg_pre, 4)) + "\t" +
                      str(round(avg_rec, 4)) + "\t" +
                      str(round(avg_f1,  4)) + "\t" +
                      str(round(avg_iou, 4)) + "\t" +
                      str(round(avg_dic, 4)) + "\n")
    result_file.close()

    print("\nMetrics saved to:", output_folder + "pothole_metrics.txt")

else:
    print("\nNo ground truth masks found. Add masks to:", gt_folder)


# Save Report Example Image

example_name = frame_files[0]
img_enhanced = cv2.imread(input_folder + example_name)
img_marked   = cv2.imread(output_folder + "marked_" + example_name)
img_mask     = cv2.imread(output_folder + "mask_"   + example_name, cv2.IMREAD_GRAYSCALE)

if img_enhanced is None:
    img_enhanced = cv2.imread(pothole_input + example_name)

if img_enhanced is not None and img_marked is not None and img_mask is not None:

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(img_enhanced, cv2.COLOR_BGR2RGB))
    plt.title("Enhanced Frame (Input)")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(img_mask, cmap='gray')
    plt.title("Segmentation Mask")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(img_marked, cv2.COLOR_BGR2RGB))
    plt.title("Marked Output - Potholes")
    plt.axis('off')

    plt.suptitle("Pothole Segmentation - Final Result")
    plt.tight_layout()
    plt.savefig(output_folder + "report_example.jpg")
    plt.show()

    print("Report example saved:", output_folder + "report_example.jpg")

    # FINAL FIXES:
#   Frames 196, 197: closing kernel increased to 31x31
#                    merges the two dark patches into one
#   Frame 198:       the two dark spots have a bright gap
#                    between them that closing cannot bridge.
#                    Solution: merge all detected contours
#                    inside the ROI into one single region
#                    using their combined convex hull.
